# Import Dataset — Chapter 3 & 4

Notebook นี้ใช้สำหรับนำเข้า Dataset ชุดใหม่ โดยใช้ชื่อ Folder:

```text
Dataset_Chapter3_4
```

## ตำแหน่งที่แนะนำใน Google Drive

```text
MyDrive/
└── Colab Notebooks/
    └── Dataset_Chapter3_4/
        ├── data/
        │   ├── metadata.csv
        │   ├── ratings.csv
        │   ├── popularity.csv
        │   ├── dataset_manifest.csv
        │   ├── images/
        │   ├── audio/
        │   └── video/
        └── figures/
```

### วิธีใช้
1. Upload `Dataset_Chapter3_4.zip` ไปที่ `MyDrive/Colab Notebooks/`
2. เปิด Notebook นี้ใน Google Colab
3. Run ตามลำดับ
4. Notebook จะ extract ZIP อัตโนมัติหากยังไม่มี Folder
5. ตรวจ schema, media files และตัวอย่างข้อมูลก่อนนำไปใช้กับ Lab บทที่ 3/4

In [14]:
# ============================================================
# Part 0 — Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

DRIVE_ROOT = Path('/content/drive/MyDrive')
COLAB_ROOT = DRIVE_ROOT / 'Colab Notebooks'

DATASET_NAME = 'Dataset_Chapter3_4'
BASE = COLAB_ROOT / DATASET_NAME
DATA_ROOT = BASE / 'data'

ZIP_FILE = COLAB_ROOT / f'{DATASET_NAME}.zip'

print('COLAB_ROOT =', COLAB_ROOT)
print('BASE       =', BASE)
print('DATA_ROOT  =', DATA_ROOT)
print('ZIP_FILE   =', ZIP_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COLAB_ROOT = /content/drive/MyDrive/Colab Notebooks
BASE       = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
DATA_ROOT  = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data
ZIP_FILE   = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4.zip


## Part 1 — Extract ZIP

ถ้า Folder `Dataset_Chapter3_4/` ยังไม่มี แต่พบ `Dataset_Chapter3_4.zip`  
Notebook จะ extract ให้อัตโนมัติ

> หาก Folder มีอยู่แล้ว Notebook จะไม่ extract ซ้ำ

In [15]:
# ============================================================
# Part 1 — Extract Dataset ZIP
# ============================================================

if not BASE.exists():

    if not ZIP_FILE.exists():
        raise FileNotFoundError(
            f'ไม่พบทั้ง Folder และ ZIP\n'
            f'Folder: {BASE}\n'
            f'ZIP: {ZIP_FILE}\n\n'
            'กรุณา Upload Dataset_Chapter3_4.zip ไปที่ MyDrive/Colab Notebooks/'
        )

    print('Extracting:', ZIP_FILE)

    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall(COLAB_ROOT)

    print('✅ Extract complete')

else:
    print('✅ Dataset folder already exists:', BASE)

print('BASE exists:', BASE.exists())
print('DATA exists:', DATA_ROOT.exists())

Extracting: /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4.zip
✅ Extract complete
BASE exists: True
DATA exists: True


## Part 2 — Check Folder Structure

ตรวจว่าไฟล์หลักและ media directories มีครบหรือไม่

In [16]:
# ============================================================
# Part 2 — Validate Folder Structure
# ============================================================

required = {
    'metadata.csv': DATA_ROOT / 'metadata.csv',
    'ratings.csv': DATA_ROOT / 'ratings.csv',
    'images': DATA_ROOT / 'images',
    'audio': DATA_ROOT / 'audio',
    'video': DATA_ROOT / 'video',
}

all_ok = True

for name, path in required.items():
    ok = path.exists()
    print(('✅' if ok else '❌'), f'{name:15s}', path)
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError(
        'โครงสร้าง Dataset_Chapter3_4 ยังไม่ครบ'
    )

print('\n✅ Required dataset structure is ready')

✅ metadata.csv    /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data/metadata.csv
✅ ratings.csv     /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data/ratings.csv
✅ images          /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data/images
✅ audio           /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data/audio
✅ video           /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data/video

✅ Required dataset structure is ready


## Part 3 — Load and Normalize Schema

Canonical schema สำหรับบทที่ 3/4:

```text
item_id
title
genre
synopsis
```

Notebook รองรับชื่อเดิม เช่น `genres` และจะ rename เป็น `genre` อัตโนมัติ

In [17]:
# ============================================================
# Part 3 — Load Dataset + Normalize Schema
# ============================================================

def normalize_items(df):
    df = df.copy()

    rename_map = {
        'movie_id': 'item_id',
        'id': 'item_id',
        'movie_title': 'title',
        'name': 'title',
        'genres': 'genre',
        'description': 'synopsis',
        'plot': 'synopsis',
        'summary': 'synopsis',
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    required_cols = ['item_id', 'title', 'genre', 'synopsis']
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise KeyError(
            f'metadata.csv ขาดคอลัมน์ {missing}\n'
            f'คอลัมน์ที่พบ: {df.columns.tolist()}'
        )

    df['item_id'] = df['item_id'].astype(str)

    for c in ['title', 'genre', 'synopsis']:
        df[c] = df[c].fillna('').astype(str)

    return df


def normalize_ratings(df):
    df = df.copy()

    rename_map = {
        'uid': 'user_id',
        'user': 'user_id',
        'iid': 'item_id',
        'movie_id': 'item_id',
        'score': 'rating',
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    required_cols = ['user_id', 'item_id', 'rating']
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise KeyError(
            f'ratings.csv ขาดคอลัมน์ {missing}\n'
            f'คอลัมน์ที่พบ: {df.columns.tolist()}'
        )

    df['user_id'] = df['user_id'].astype(str)
    df['item_id'] = df['item_id'].astype(str)
    df['rating'] = pd.to_numeric(df['rating'], errors='raise')

    return df


items = normalize_items(
    pd.read_csv(DATA_ROOT / 'metadata.csv')
)

ratings = normalize_ratings(
    pd.read_csv(DATA_ROOT / 'ratings.csv')
)

popularity_file = DATA_ROOT / 'popularity.csv'
popularity = (
    pd.read_csv(popularity_file)
    if popularity_file.exists()
    else None
)

print('✅ Dataset loaded')
print('Items   :', len(items))
print('Users   :', ratings['user_id'].nunique())
print('Ratings :', len(ratings))
print()
print('Item columns:', items.columns.tolist())
print('Rating columns:', ratings.columns.tolist())

display(items.head(12))
display(ratings.head(10))

✅ Dataset loaded
Items   : 12
Users   : 8
Ratings : 32

Item columns: ['item_id', 'title', 'year', 'genre', 'image_brightness', 'image_warm', 'image_cartoon', 'audio_energy', 'audio_melody', 'audio_dark', 'video_change', 'video_cuts', 'video_animation', 'synopsis']
Rating columns: ['user_id', 'item_id', 'rating']


,item_id,title,year,genre,image_brightness,image_warm,image_cartoon,audio_energy,audio_melody,audio_dark,video_change,video_cuts,video_animation,synopsis
0,i1,Toy Story 5,2026,Animation|Family,0.9,0.6,0.9,0.5,0.7,0.1,0.6,0.5,1.0,A cowboy toy and a space ranger toy lead the t...
1,i2,Inside Out 2,2024,Animation|Family,0.8,0.8,0.9,0.5,1.0,0.2,0.5,0.4,1.0,"Inside the mind of a teenage girl, five emotio..."
2,i3,The Wild Robot,2024,Animation|Sci-Fi,0.7,0.5,0.8,0.3,0.8,0.4,0.4,0.3,1.0,A shipwrecked robot boots up on a wild island ...
3,i4,It Ends with Us,2024,Drama|Romance,0.5,0.7,0.2,0.3,0.8,0.3,0.4,0.3,0.0,A florist falls in love with a charming surgeo...
4,i5,Wicked,2024,Fantasy|Musical,0.7,0.6,0.5,0.4,1.0,0.1,0.3,0.4,0.0,A green-skinned woman with a gift for magic jo...
5,i6,Bridget Jones: Mad About the Boy,2025,Drama|Romance,0.6,0.8,0.3,0.2,0.8,0.1,0.2,0.2,0.0,A widowed mother of two children keeps talking...
6,i7,Ballerina,2025,Action|Thriller,0.2,0.3,0.1,0.9,0.2,0.8,0.9,0.9,0.0,A ballerina trained as an assassin by a secret...
7,i8,Furiosa: A Mad Max Saga,2024,Action|Sci-Fi,0.6,0.9,0.3,1.0,0.3,0.7,1.0,1.0,0.0,"Snatched from a fertile place as a child, a su..."
8,i9,Conclave,2024,Drama|Thriller,0.4,0.3,0.1,0.3,0.5,0.6,0.3,0.4,0.0,"After the death of the pope, a cardinal must r..."
9,i10,Dune: Part Two,2024,Action|Sci-Fi,0.5,0.8,0.1,0.8,0.3,0.8,0.7,0.6,0.0,"On a desert planet, a young heir of a fallen d..."


,user_id,item_id,rating
0,u1,i1,5
1,u1,i2,5
2,u1,i3,4
3,u1,i4,1
4,u1,i6,1
5,u2,i4,5
6,u2,i5,4
7,u2,i6,5
8,u2,i1,2
9,u3,i10,5


## Part 4 — Attach Media Paths

ใช้ `item_id` เชื่อมกับไฟล์ Image / Audio / Video  
รองรับนามสกุลพื้นฐานที่ใช้ใน Lab

In [18]:
# ============================================================
# Part 4 — Build Media Paths
# ============================================================

IMAGE_DIR = DATA_ROOT / 'images'
AUDIO_DIR = DATA_ROOT / 'audio'
VIDEO_DIR = DATA_ROOT / 'video'


def find_media(folder, item_id, extensions):

    for ext in extensions:
        p = folder / f'{item_id}{ext}'

        if p.exists():
            return str(p)

    return None


items['image_path'] = items['item_id'].apply(
    lambda x: find_media(
        IMAGE_DIR,
        x,
        ['.png', '.jpg', '.jpeg', '.webp']
    )
)

items['audio_path'] = items['item_id'].apply(
    lambda x: find_media(
        AUDIO_DIR,
        x,
        ['.wav', '.mp3']
    )
)

items['video_path'] = items['item_id'].apply(
    lambda x: find_media(
        VIDEO_DIR,
        x,
        ['.mp4', '.avi', '.mov']
    )
)

display(
    items[
        [
            'item_id',
            'title',
            'genre',
            'image_path',
            'audio_path',
            'video_path'
        ]
    ]
)

,item_id,title,genre,image_path,audio_path,video_path
0,i1,Toy Story 5,Animation|Family,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
1,i2,Inside Out 2,Animation|Family,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
2,i3,The Wild Robot,Animation|Sci-Fi,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
3,i4,It Ends with Us,Drama|Romance,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
4,i5,Wicked,Fantasy|Musical,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
5,i6,Bridget Jones: Mad About the Boy,Drama|Romance,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
6,i7,Ballerina,Action|Thriller,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
7,i8,Furiosa: A Mad Max Saga,Action|Sci-Fi,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
8,i9,Conclave,Drama|Thriller,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
9,i10,Dune: Part Two,Action|Sci-Fi,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...


## Part 5 — Media Validation

สำหรับบทที่ 3 ใช้เฉพาะ Text ได้  
สำหรับบทที่ 4 ควรตรวจว่า Image / Audio / Video มีครบ

In [19]:
# ============================================================
# Part 5 — Validate Media Availability
# ============================================================

items['image_exists'] = items['image_path'].notna()
items['audio_exists'] = items['audio_path'].notna()
items['video_exists'] = items['video_path'].notna()

summary = pd.DataFrame({
    'modality': ['Text', 'Image', 'Audio', 'Video'],
    'available_items': [
        len(items),
        int(items['image_exists'].sum()),
        int(items['audio_exists'].sum()),
        int(items['video_exists'].sum()),
    ],
    'total_items': [len(items)] * 4
})

display(summary)

display(
    items[
        [
            'item_id',
            'title',
            'image_exists',
            'audio_exists',
            'video_exists'
        ]
    ]
)

,modality,available_items,total_items
0,Text,12,12
1,Image,12,12
2,Audio,12,12
3,Video,12,12


,item_id,title,image_exists,audio_exists,video_exists
0,i1,Toy Story 5,True,True,True
1,i2,Inside Out 2,True,True,True
2,i3,The Wild Robot,True,True,True
3,i4,It Ends with Us,True,True,True
4,i5,Wicked,True,True,True
5,i6,Bridget Jones: Mad About the Boy,True,True,True
6,i7,Ballerina,True,True,True
7,i8,Furiosa: A Mad Max Saga,True,True,True
8,i9,Conclave,True,True,True
9,i10,Dune: Part Two,True,True,True


## Part 6 — Chapter 3 Text Preview

สร้าง `text_content = genre + synopsis`  
เพื่อใช้กับ TF-IDF, Word2Vec, BERT, SBERT

In [20]:
# ============================================================
# Part 6 — Chapter 3 Text Input
# ============================================================

items['text_content'] = (
    items['genre'].fillna('')
    + ' '
    + items['synopsis'].fillna('')
).str.strip()

display(
    items[
        [
            'item_id',
            'title',
            'text_content'
        ]
    ].head()
)

print('\n✅ Chapter 3 text input is ready')

,item_id,title,text_content
0,i1,Toy Story 5,Animation|Family A cowboy toy and a space rang...
1,i2,Inside Out 2,Animation|Family Inside the mind of a teenage ...
2,i3,The Wild Robot,Animation|Sci-Fi A shipwrecked robot boots up ...
3,i4,It Ends with Us,Drama|Romance A florist falls in love with a c...
4,i5,Wicked,Fantasy|Musical A green-skinned woman with a g...



✅ Chapter 3 text input is ready


## Part 7 — Dataset Summary for Chapter 3 & 4

หลังจาก Run ถึงจุดนี้ สามารถใช้ตัวแปรต่อไปนี้กับ Notebook อื่นได้โดยตรง:

```python
BASE
DATA_ROOT
items
ratings
popularity
```

### Chapter 3
ใช้:
- `items["text_content"]`
- `ratings`

### Chapter 4
ใช้:
- `items["image_path"]`
- `items["audio_path"]`
- `items["video_path"]`
- `ratings`

In [21]:
# ============================================================
# Part 7 — Final Dataset Summary
# ============================================================

print('=' * 60)
print('Dataset_Chapter3_4 READY')
print('=' * 60)

print('BASE      :', BASE)
print('DATA_ROOT :', DATA_ROOT)
print('Items     :', len(items))
print('Users     :', ratings['user_id'].nunique())
print('Ratings   :', len(ratings))
print('Images    :', items['image_exists'].sum(), '/', len(items))
print('Audio     :', items['audio_exists'].sum(), '/', len(items))
print('Video     :', items['video_exists'].sum(), '/', len(items))

print('\n✅ Ready for Chapter 3 and Chapter 4 Labs')

Dataset_Chapter3_4 READY
BASE      : /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
DATA_ROOT : /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data
Items     : 12
Users     : 8
Ratings   : 32
Images    : 12 / 12
Audio     : 12 / 12
Video     : 12 / 12

✅ Ready for Chapter 3 and Chapter 4 Labs
